In [2]:
import re
import numpy as np
import matplotlib.pyplot as plt

def parse_multiple_nz_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    pattern = r"\(1x1x(\d+)\).*?--- \[ABD\] Full Matrix.*?---\n\[\[(.*?)\]\]"
    matches = re.findall(pattern, content, re.DOTALL)

    nz_list = []
    matrices = []

    for nz_str, mat_str in matches:
        nz = int(nz_str)
        mat_str_clean = mat_str.replace('[', ' ').replace(']', ' ')
        vals = [float(x) for x in mat_str_clean.split()]

        if len(vals) == 36:
            mat = np.array(vals).reshape((6, 6))
            nz_list.append(nz)
            matrices.append(mat)

    return np.array(nz_list), np.array(matrices)

def main():
    # ================= 1. 数据读取与定义 =================
    file_path = r'E:\~Paper\2026_ThinWalledHomo\5_Figures\Fig3.2\TPMS multiple.txt'
    nz_list, matrices = parse_multiple_nz_file(file_path)

    if len(nz_list) == 0:
        print("未能提取到数据，请检查文件路径。")
        return

    # 目标 3D-VH 理论解析极限
    mat_3d = np.array([
        [674.00,  226.65,    0.0,      0.0,      0.0,      0.0],
        [226.65,  674.00,    0.0,      0.0,      0.0,      0.0],
        [  0.0,     0.0,   245.31,     0.0,      0.0,      0.0],
        [  0.0,     0.0,     0.0,   5616.63,  1888.76,     0.0],
        [  0.0,     0.0,     0.0,   1888.76,  5616.63,     0.0],
        [  0.0,     0.0,     0.0,      0.0,      0.0,   2044.26]
    ])

    # 提取 A 矩阵分量 (11, 22, 12, 66)
    A11_2d, A22_2d = matrices[:, 0, 0], matrices[:, 1, 1]
    A12_2d, A66_2d = matrices[:, 0, 1], matrices[:, 2, 2]

    # 提取 D 矩阵分量 (11, 22, 12, 66)
    D11_2d, D22_2d = matrices[:, 3, 3], matrices[:, 4, 4]
    D12_2d, D66_2d = matrices[:, 3, 4], matrices[:, 5, 5]

    # 提取 3D 极限
    A11_3d, A22_3d, A12_3d, A66_3d = mat_3d[0,0], mat_3d[1,1], mat_3d[0,1], mat_3d[2,2]
    D11_3d, D22_3d, D12_3d, D66_3d = mat_3d[3,3], mat_3d[4,4], mat_3d[3,4], mat_3d[5,5]

    # 计算归一化刚度 (Normalized Stiffness) -> 全部趋近于 1.0
    norm_A11, norm_A22 = A11_2d / A11_3d, A22_2d / A22_3d
    norm_A12, norm_A66 = A12_2d / A12_3d, A66_2d / A66_3d

    norm_D11, norm_D22 = D11_2d / D11_3d, D22_2d / D22_3d
    norm_D12, norm_D66 = D12_2d / D12_3d, D66_2d / D66_3d

    # ================= 2. 全局字体与学术排版设置 =================
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['font.size'] = 9
    plt.rcParams['mathtext.fontset'] = 'stix'

    # 定义四种高级莫兰迪配色方案
    c_11 = '#4A6B8A' # 深蓝灰
    c_22 = '#C27471' # 砖红
    c_12 = '#E2B879' # 莫兰迪芥末黄 (用来代表泊松耦合)
    c_66 = '#6A8074' # 豆沙绿 (用来代表剪切/扭转)

    # 稍微拉宽一点画布，避免图例遮挡曲线
    fig, axes = plt.subplots(1, 2, figsize=(16/2.54, 6/2.54))
    plt.subplots_adjust(wspace=0.35, left=0.08, right=0.98, bottom=0.25, top=0.88)

    # ------------------ (a) 面内刚度收敛 (A Matrix) ------------------
    ax1 = axes[0]
    ax1.axhline(1.0, color='gray', linestyle='--', linewidth=1.2, zorder=1)

    ax1.plot(nz_list, norm_A11, marker='o', ms=3, color=c_11, lw=1, label='$A_{11}$', zorder=2)
    ax1.plot(nz_list, norm_A22, marker='s', ms=3, color=c_22, lw=1, label='$A_{22}$', zorder=2)
    ax1.plot(nz_list, norm_A12, marker='^', ms=3, color=c_12, lw=1, label='$A_{12}$', zorder=2)
    ax1.plot(nz_list, norm_A66, marker='d', ms=3, color=c_66, lw=1, label='$A_{66}$', zorder=2)

    ax1.set_xlabel('Cell Count in Thickness ($N_z$)')
    ax1.set_ylabel('Normalized In-plane Stiffness')
    ax1.set_xlim(0.5, 6.5)
    ax1.set_ylim(0.5, 1.05)
    ax1.set_xticks(nz_list)
    # 调整图例列数，避免遮挡折线
    ax1.legend(loc='lower right', frameon=False, fontsize=8, ncol=2)
    ax1.set_title('In-plane Stiffness ($A_{ij}$)', fontsize=10, pad=6)

    # ------------------ (b) 面外弯曲刚度收敛 (D Matrix) ------------------
    ax2 = axes[1]
    ax2.axhline(1.0, color='gray', linestyle='--', linewidth=1.2, zorder=1)

    ax2.plot(nz_list, norm_D11, marker='o', ms=3, color=c_11, lw=1, label='$D_{11}$', zorder=2)
    ax2.plot(nz_list, norm_D22, marker='s', ms=3, color=c_22, lw=1, label='$D_{22}$', zorder=2)
    ax2.plot(nz_list, norm_D12, marker='^', ms=3, color=c_12, lw=1, label='$D_{12}$', zorder=2)
    ax2.plot(nz_list, norm_D66, marker='d', ms=3, color=c_66, lw=1, label='$D_{66}$', zorder=2)

    ax2.set_xlabel('Cell Count in Thickness ($N_z$)')
    ax2.set_ylabel('Normalized Bending Stiffness')
    ax2.set_xlim(0.5, 6.5)
    ax2.set_ylim(0.2, 1.05)
    ax2.set_xticks(nz_list)
    ax2.legend(loc='lower right', frameon=False, fontsize=8, ncol=2)
    ax2.set_title('Out-of-plane Bending ($D_{ij}$)', fontsize=10, pad=6)

    # ================= 3. 输出保存 =================
    output_name = 'Stiffness_Full_Convergence_Gyroid.png'
    plt.savefig(output_name, dpi=400, bbox_inches='tight')
    plt.close()

    print(f"完整分量收敛图已生成: {output_name}")

if __name__ == "__main__":
    main()

完整分量收敛图已生成: Stiffness_Full_Convergence_Gyroid.png


In [2]:
"""
tools/export_vtu_models.py

A utility script to generate multiple array configurations of TPMS voxel grids
and export them as ParaView-compatible .vtu (Unstructured Grid) files.
"""

import os
import time
import numpy as np

try:
    import pyvista as pv
except ImportError:
    raise ImportError("Please install pyvista to export VTU files: pip install pyvista")

from utils.tpms_generator import generate_tpms_voxel_grid


def save_voxel_to_vtu(voxel_array, filename, pitch=1.0):
    """
    Convert a 3D binary numpy array to a PyVista UnstructuredGrid and save as .vtu.
    Filters out void regions (0s) to keep the file size minimal.

    Parameters:
    -----------
    voxel_array : numpy.ndarray
        3D binary array (1 for solid, 0 for void) with (x, y, z) indexing.
    filename : str
        The output file path.
    pitch : float
        The physical size of a single voxel.
    """
    nx, ny, nz = voxel_array.shape

    # 1. Create a uniform rectilinear grid (ImageData)
    # The dimensions are number of nodes, which is number of cells + 1
    grid = pv.ImageData()
    grid.dimensions = (nx + 1, ny + 1, nz + 1)
    grid.spacing = (pitch, pitch, pitch)
    grid.origin = (0.0, 0.0, 0.0)

    # 2. Assign voxel values to cell data
    # VTK expects arrays flattened in Fortran order ('F')
    grid.cell_data["Material"] = voxel_array.flatten(order="F")

    # 3. Extract only solid elements to create an UnstructuredGrid
    # This removes the void spaces (0) and significantly reduces file size
    solid_grid = grid.threshold(0.5, scalars="Material")

    # 4. Save to disk
    solid_grid.save(filename)
    print(f"      -> Saved successfully: {filename} ({solid_grid.n_cells} solid elements)")


def batch_generate_and_export():
    # Define output directory
    out_dir = "vtu_outputs"
    os.makedirs(out_dir, exist_ok=True)

    # TPMS Parameters
    tpms_type = 'Gyroid'
    relative_density = 0.15
    is_sheet = True

    # Define the requested target configurations (Nx, Ny, Nz)
    target_configs = [
        (1, 1, 1),
        (1, 1, 2),
        (1, 1, 3),
        (1, 1, 4),
    ]

    print(f"Starting batch generation for {tpms_type} Sheet models...\n")

    for (Nx, Ny, Nz) in target_configs:
        # Dynamic resolution adjustment to prevent Out-Of-Memory (OOM) errors
        # For a 10x10x10 array, we use a lower resolution per cell.
        total_cells = Nx * Ny * Nz
        res = 64

        print(f"[{Nx}x{Ny}x{Nz}] Configuration (res={res}/cell):")

        t0 = time.time()
        # 1. Generate voxel grid
        voxel_grid = generate_tpms_voxel_grid(
            tpms_type=tpms_type,
            Nx=Nx, Ny=Ny, Nz=Nz,
            resolution=res,
            relative_density=relative_density,
            is_sheet=is_sheet
        )
        t1 = time.time()
        print(f"      -> Generation time: {t1 - t0:.2f} s | Shape: {voxel_grid.shape}")

        # 2. Export to VTU
        # Normalize the voxel pitch so that 1 unit cell = 10.0 mm physically
        unit_cell_size = 10.0
        voxel_pitch = unit_cell_size / res

        filename = os.path.join(out_dir, f"{tpms_type}_Sheet_{Nx}x{Ny}x{Nz}.vtu")

        t2 = time.time()
        save_voxel_to_vtu(voxel_grid, filename, pitch=voxel_pitch)
        t3 = time.time()
        print(f"      -> Export time: {t3 - t2:.2f} s")
        print("-" * 60)

    print("\nAll models exported successfully. You can open them directly in ParaView.")


if __name__ == "__main__":
    batch_generate_and_export()

Starting batch generation for Gyroid Sheet models...

[1x1x1] Configuration (res=64/cell):
      -> Generation time: 0.01 s | Shape: (64, 64, 64)
      -> Saved successfully: vtu_outputs\Gyroid_Sheet_1x1x1.vtu (39323 solid elements)
      -> Export time: 0.06 s
------------------------------------------------------------
[1x1x2] Configuration (res=64/cell):
      -> Generation time: 0.03 s | Shape: (64, 64, 128)
      -> Saved successfully: vtu_outputs\Gyroid_Sheet_1x1x2.vtu (78666 solid elements)
      -> Export time: 0.12 s
------------------------------------------------------------
[1x1x3] Configuration (res=64/cell):
      -> Generation time: 0.04 s | Shape: (64, 64, 192)
      -> Saved successfully: vtu_outputs\Gyroid_Sheet_1x1x3.vtu (117969 solid elements)
      -> Export time: 0.17 s
------------------------------------------------------------
[1x1x4] Configuration (res=64/cell):
      -> Generation time: 0.05 s | Shape: (64, 64, 256)
      -> Saved successfully: vtu_outputs\Gy

# 收敛性分析


In [3]:
import re
import numpy as np
import matplotlib.pyplot as plt

def parse_convergence_file(file_path):
    # 读取日志文件
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 使用正则表达式匹配 res 和对应的 6x6 矩阵区域
    pattern = r"at res=(\d+)\.\.\..*?--- \[ABD\] Full Matrix.*?---\n\[\[(.*?)\]\]"
    matches = re.findall(pattern, content, re.DOTALL)

    res_list = []
    matrices = []

    for res_str, mat_str in matches:
        res = int(res_str)
        # 清洗矩阵字符串，去除中间可能出现的换行符和括号
        mat_str_clean = mat_str.replace('[', ' ').replace(']', ' ')

        # 转换为 float，Python 默认的 float() 能够完美解析科学计数法(如 -9.97e+08)
        vals = [float(x) for x in mat_str_clean.split()]

        # 确保成功提取了完整的 36 个分量
        if len(vals) == 36:
            mat = np.array(vals).reshape((6, 6))
            res_list.append(res)
            matrices.append(mat)

    return np.array(res_list), np.array(matrices)

def main():
    # ================= 1. 数据读取与定义 =================
    # 请确保 txt 文件路径正确
    file_path = r'E:\~Paper\2026_ThinWalledHomo\5_Figures\Convergence.txt'
    res_list, matrices = parse_convergence_file(file_path)

    if len(res_list) == 0:
        print("未能提取到数据，请检查文件路径或格式。")
        return

    # 提取 A11 和 D11 分量 (A11在0,0位置，D11在3,3位置)
    A11 = matrices[:, 0, 0]
    D11 = matrices[:, 3, 3]

    # ================= 2. 全局字体与学术排版设置 =================
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman']
    plt.rcParams['font.size'] = 9
    plt.rcParams['mathtext.fontset'] = 'stix'

    # 定义高级莫兰迪配色方案
    c_11 = '#4A6B8A' # 深蓝灰
    c_22 = '#C27471' # 砖红

    # 创建 1x2 的子图排版
    fig, axes = plt.subplots(1, 2, figsize=(16/2.54, 6/2.54))
    plt.subplots_adjust(wspace=0.35, left=0.08, right=0.98, bottom=0.25, top=0.88)

    # ------------------ (a) 面内刚度收敛 (A11) ------------------
    ax1 = axes[0]
    ax1.plot(res_list, A11, marker='o', ms=3, color=c_11, lw=1, label='$A_{11}$', zorder=2)

    ax1.set_xlabel('Mesh Resolution ($N$)')
    ax1.set_ylabel('In-plane Stiffness $A_{11}$ (MPa$\\cdot$mm)')
    ax1.set_xticks(res_list[::2]) # 标签避免过密，每隔一个显示
    ax1.legend(loc='best', frameon=False, fontsize=8)
    ax1.set_title('Convergence of $A_{11}$', fontsize=10, pad=6)

    # 【智能滤除爆炸值】：找到合理的收敛区间以自适应 Y 轴范围
    stable_A11 = A11[np.abs(A11) < 1e4]
    if len(stable_A11) > 0:
        ymin, ymax = np.min(stable_A11) * 0.95, np.max(stable_A11) * 1.05
        ax1.set_ylim(ymin, ymax)

    # ------------------ (b) 面外弯曲刚度收敛 (D11) ------------------
    ax2 = axes[1]
    ax2.plot(res_list, D11, marker='s', ms=3, color=c_22, lw=1, label='$D_{11}$', zorder=2)

    ax2.set_xlabel('Mesh Resolution ($N$)')
    ax2.set_ylabel('Bending Stiffness $D_{11}$ (MPa$\\cdot$mm$^3$)')
    ax2.set_xticks(res_list[::2])
    ax2.legend(loc='best', frameon=False, fontsize=8)
    ax2.set_title('Convergence of $D_{11}$', fontsize=10, pad=6)

    # 【智能滤除爆炸值】
    stable_D11 = D11[np.abs(D11) < 1e5]
    if len(stable_D11) > 0:
        ymin, ymax = np.min(stable_D11) * 0.95, np.max(stable_D11) * 1.05
        ax2.set_ylim(ymin, ymax)

    # ================= 3. 输出保存 =================
    output_name = 'Convergence_A11_D11.png'
    plt.savefig(output_name, dpi=400, bbox_inches='tight')
    plt.close()

    print(f"收敛图已成功生成: {output_name}")
    print(f"共提取了 {len(res_list)} 个数据点。")

if __name__ == "__main__":
    main()

收敛图已成功生成: Convergence_A11_D11.png
共提取了 22 个数据点。
